# Descarga de los datos agrícolas

Esta libreta documenta la obtención y preparación de los datos agrícolas
utilizados para estudiar los factores asociados con el éxito de la
producción agrícola en los municipios de México entre **2003 y 2025**.

Los archivos agrícolas se obtienen anualmente de la base de datos abiertos de
la Secretaría de Agricultura y posteriormente se integran en un único
conjunto histórico.

Debido a que la estructura de los archivos presenta algunos cambios a
lo largo del periodo de estudio, el procesamiento incluye una etapa de
normalización para mantener un esquema consistente entre los años.

La lógica de descarga y transformación se encuentra implementada en el
paquete `proyecto_agronomia`. Esta libreta permite ejecutar, inspeccionar
y validar el proceso de manera reproducible.

### Periodo de estudio

**2003–2025**

### Flujo de trabajo

1. Descargar los archivos agrícolas anuales.
2. Verificar la disponibilidad de los archivos.
3. Inspeccionar un archivo agrícola.
4. Normalizar las diferencias históricas del esquema.
5. Construir el conjunto agrícola histórico.
6. Generar variables derivadas e indicadores de calidad.
7. Validar el conjunto resultante.

In [ ]:
import pandas as pd

from proyecto_agronomia.config import (
    AGRICULTURA_RAW_DIR,
    ANIO_INICIAL,
    ANIO_FINAL,
    ANIOS,
)

from proyecto_agronomia.dataset import (
    descargar_datos_agricolas,
    descargar_diccionario_agricola,
)

## 1. Descarga de los datos agrícolas

La fuente agrícola proporciona un archivo independiente para cada año
del periodo de estudio.

Para construir la serie histórica se consideran los años comprendidos
entre **2003 y 2025**.

Los archivos originales se almacenan en:

`data/raw/agricultura/`

Los datos de esta carpeta representan la información original obtenida
de la fuente y no son modificados directamente durante las etapas
posteriores del procesamiento.

La función de descarga verifica si cada archivo ya existe localmente.
Cuando un archivo está disponible, se reutiliza en lugar de volver a
descargarlo.

In [3]:
print(
    f"Periodo de estudio: "
    f"{ANIO_INICIAL}-{ANIO_FINAL}"
)

print(
    f"Años considerados: "
    f"{len(ANIOS)}"
)

print(
    f"Directorio de datos originales: "
    f"{AGRICULTURA_RAW_DIR}"
)

Periodo de estudio: 2003-2025
Años considerados: 23
Directorio de datos originales: /home/dusty/Documents/Repositories/proyecto_agronomia/data/raw/agricultura


## 2. Ejecución de la descarga

Se utiliza la función `descargar_datos_agricolas()` del módulo
`proyecto_agronomia.dataset`.

La función recorre los años del periodo de estudio y conserva los
archivos que ya se encuentran disponibles localmente, por lo que la
operación puede ejecutarse nuevamente sin descargar innecesariamente
los mismos datos.

In [4]:
archivos_agricolas = descargar_datos_agricolas(
    ANIOS
)

print(
    f"Archivos disponibles: "
    f"{len(archivos_agricolas)}"
)

Datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

2026-11-14 11:10:36.192 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:51 - [2003] Descargando...


Datos agrícolas:   4%|▍         | 1/23 [00:02<00:43,  1.99s/it]

2026-11-14 11:10:38.178 | SUCCESS  | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2003] Descargado (5.80 MB)
2026-11-14 11:10:38.192 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:51 - [2004] Descargando...


Datos agrícolas:   9%|▊         | 2/23 [00:08<01:35,  4.55s/it]

2026-11-14 11:10:44.512 | SUCCESS  | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2004] Descargado (5.89 MB)
2026-11-14 11:10:44.521 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:51 - [2005] Descargando...


... (38 outputs left out)

Datos agrícolas:  96%|█████████▌| 22/23 [01:23<00:04,  4.50s/it]

2026-11-14 11:11:59.704 | SUCCESS  | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2024] Descargado (5.68 MB)
2026-11-14 11:11:59.712 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:51 - [2025] Descargando...


Datos agrícolas: 100%|██████████| 23/23 [01:24<00:00,  3.69s/it]

2026-11-14 11:12:01.026 | SUCCESS  | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2025] Descargado (5.50 MB)
Archivos disponibles: 23


### 2.1. Verificación de los archivos

Antes de iniciar el procesamiento se verifica que exista un archivo
para cada uno de los años considerados en el estudio.

Esta validación permite detectar años faltantes antes de construir el
conjunto histórico. Por defecto esta de 2003 a 2025, entonces esperamos 23 archivos

In [10]:
disponibilidad = pd.DataFrame({
        "Anio": ANIOS,
        "Archivo correspondiente": [
            f"agricola_{anio}.csv"
            for anio in ANIOS if (AGRICULTURA_RAW_DIR / f"agricola_{anio}.csv").exists()
        ]
    }
)

disponibilidad

,Anio,Archivo correspondiente
0,2003,agricola_2003.csv
1,2004,agricola_2004.csv
2,2005,agricola_2005.csv
3,2006,agricola_2006.csv
4,2007,agricola_2007.csv
5,2008,agricola_2008.csv
6,2009,agricola_2009.csv
7,2010,agricola_2010.csv
8,2011,agricola_2011.csv
9,2012,agricola_2012.csv


In [13]:
print(
    f"Archivos esperados: "
    f"{len(ANIOS)}"
)

print(
    f"Archivos disponibles: "
    f"{disponibilidad.shape[0]}"
)

print(
    f"Archivos faltantes: "
    f"{ ANIO_FINAL - ANIO_INICIAL + 1 - disponibilidad.shape[0]}"
)

Archivos esperados: 23
Archivos disponibles: 23
Archivos faltantes: 0


### Diccionario de Datos

Siempre al descargar los datos. Debemos adjuntar su Diccionario de Datos.

In [22]:
ruta_diccionario = descargar_diccionario_agricola()

print(
    f"Diccionario disponible en: "
    f"{ruta_diccionario}"
)

2026-11-14 12:25:39.285 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_diccionario_agricola:107 - El diccionario agrícola ya existe.
Diccionario disponible en: /home/dusty/Documents/Repositories/proyecto_agronomia/references/diccionario_datos_agricultura_dgsiap.xlsx
